# QuoteCaster: text-only anonymous speaker test
Local model setup, then **two independent inference cells**: one page and all pages. Select a sequence from labels.jsonl. Only its text enters the model; GT speakers, images and project entity IDs are never used.

QuoteCaster is an encoder: raw output is a CLS embedding per contextualized quote, not generated entity labels. We apply cosine agglomerative clustering to obtain arbitrary E1/E2 labels. These are local to each run and have no association with our task's characters or with labels in the other cell. The clustering threshold is exploratory, not a calibrated same-speaker probability.

Reference: [QuoteCaster model card](https://huggingface.co/aNameNobodyChose/quote-caster-encoder). Run in godfather.venv after the local download finishes. No model/data downloads or training occur here.

## Setup — local model, selected sequence and shared helpers
Change MODEL_PATH if the downloaded folder has another name. CONTEXT_BOXES=2 uses the previous two and next two texts in the selected scope. In all-pages mode that context may cross page boundaries. Every selected text is encoded individually, then all its embeddings are clustered together.

In [8]:
import os
os.environ['HF_HUB_OFFLINE']='1';os.environ['TRANSFORMERS_OFFLINE']='1'
import json
from pathlib import Path
import numpy as np
import pandas as pd
import torch
from transformers import AutoTokenizer,AutoModel
from sklearn.cluster import AgglomerativeClustering
from IPython.display import display

MODEL_PATH=Path('C:/MODELS/quote-caster-encoder')
LABELS_PATH=Path('C:/Users/ESHAAN/BYTE/Applied-Task/Magi/TaskData/development/labels.jsonl')
SEQUENCE_ID='seq_2fae297a18af2a57'
PAGE_NUMBER=1  # Used only by the one-page cell below; 1-based.
CONTEXT_BOXES=2  # Earlier/later texts around each target, never speaker labels.
CLUSTER_DISTANCE=0.20  # Cosine distance: lower gives more groups. Adjust after review.
N_SPEAKERS=None  # Optional explicit group count; never inferred from GT speakers.
MAX_TOKENS=512
BATCH_SIZE=16
DEVICE='cuda' if torch.cuda.is_available() else 'cpu'

# Only look for the specifically downloaded QuoteCaster model, not other models.
if not MODEL_PATH.is_dir():
    candidates=[p for p in Path('C:/MODELS').iterdir() if p.is_dir()
        and ('quotecaster' in p.name.lower().replace('-','').replace('_',''))]
    if len(candidates)==1:MODEL_PATH=candidates[0]
assert (MODEL_PATH/'config.json').is_file(),f'Finish downloading QuoteCaster or set MODEL_PATH: {MODEL_PATH}'
assert LABELS_PATH.is_file(),LABELS_PATH
assert CONTEXT_BOXES>=0 and 0<CLUSTER_DISTANCE<=2
sequence=None
for line in LABELS_PATH.read_text(encoding='utf-8').splitlines():
    if line.strip():
        candidate=json.loads(line)
        if candidate['sequence_id']==SEQUENCE_ID.strip():
            assert sequence is None,'Duplicate sequence ID in labels.jsonl'
            sequence={'sequence_id':candidate['sequence_id'],
                'pages':[[{'text':entry['text']} for entry in page] for page in candidate['pages']]}
assert sequence is not None,f'Unknown sequence: {SEQUENCE_ID}'
# Strip GT speakers immediately: only text remains in the object used below.
tokenizer=AutoTokenizer.from_pretrained(str(MODEL_PATH),local_files_only=True)
model=AutoModel.from_pretrained(str(MODEL_PATH),local_files_only=True).to(DEVICE).eval()
model.requires_grad_(False)
assert model.config.hidden_size==768,'Expected the QuoteCaster BERT encoder'
TOKEN_LIMIT=min(MAX_TOKENS,model.config.max_position_embeddings)
assert TOKEN_LIMIT>=4


def page_text_records(page_numbers):
    records=[]
    for page_number in page_numbers:
        assert 1<=page_number<=len(sequence['pages']),page_number
        for index,entry in enumerate(sequence['pages'][page_number-1]):
            records.append(dict(entry_id=f'PG{page_number}:T{index+1}',page=page_number,
                text_index=index,text=str(entry['text'])))
    return records


def make_quote_inputs(records):
    items=[]
    for index,record in enumerate(records):
        previous=[r['text'] for r in records[max(0,index-CONTEXT_BOXES):index]]
        following=[r['text'] for r in records[index+1:index+1+CONTEXT_BOXES]]
        context=' '.join(previous+following)
        context_ids=tokenizer.encode(context,add_special_tokens=False)
        quote_ids=tokenizer.encode(record['text'],add_special_tokens=False) or [tokenizer.unk_token_id]
        # Match context [SEP] quote format, reserving target space before truncating context.
        quote_ids=quote_ids[:TOKEN_LIMIT-3]
        budget=TOKEN_LIMIT-3-len(quote_ids)
        context_ids=context_ids[-budget:] if budget else []
        ids=[tokenizer.cls_token_id]+context_ids+[tokenizer.sep_token_id]+quote_ids
        items.append(ids)
    return items


@torch.inference_mode()
def encode_quotes(records):
    vectors=[];items=make_quote_inputs(records)
    for start in range(0,len(items),BATCH_SIZE):
        batch=items[start:start+BATCH_SIZE];length=max(map(len,batch))
        ids=torch.full((len(batch),length),tokenizer.pad_token_id,dtype=torch.long,device=DEVICE)
        attention=torch.zeros_like(ids)
        for i,tokens in enumerate(batch):
            ids[i,:len(tokens)]=torch.tensor(tokens,device=DEVICE);attention[i,:len(tokens)]=1
        output=model(input_ids=ids,attention_mask=attention,token_type_ids=torch.zeros_like(ids))
        vectors.append(output.last_hidden_state[:,0,:].float().cpu())
    raw=torch.cat(vectors).numpy() if vectors else np.empty((0,768),dtype=np.float32)
    assert np.isfinite(raw).all()
    return raw


def group_quote_vectors(raw):
    if len(raw)==0:return np.empty(0,dtype=int)
    if len(raw)==1:return np.zeros(1,dtype=int)
    norms=np.linalg.norm(raw,axis=1,keepdims=True)
    assert (norms>0).all(),'Zero-length model embedding'
    vectors=raw/norms
    if N_SPEAKERS is not None:
        assert 1<=int(N_SPEAKERS)<=len(vectors)
        clustering=AgglomerativeClustering(n_clusters=int(N_SPEAKERS),metric='cosine',linkage='average')
    else:
        clustering=AgglomerativeClustering(n_clusters=None,distance_threshold=CLUSTER_DISTANCE,
            metric='cosine',linkage='average')
    return clustering.fit_predict(vectors)


def infer_entities(records):
    raw=encode_quotes(records)
    clusters=group_quote_vectors(raw)
    names={};rows=[]
    for record,cluster in zip(records,clusters):
        label=names.setdefault(int(cluster),f'E{len(names)+1}')
        rows.append(dict(record,entity=label))
    results=pd.DataFrame(rows,columns=['entry_id','page','text_index','text','entity'])
    display(results)
    print('Texts:',len(records),'| Anonymous groups:',len(names),'| Raw encoder output:',raw.shape)
    for entity in results['entity'].unique():
        print('\n'+entity)
        for row in results.loc[results['entity']==entity].itertuples():print(row.entry_id+': '+row.text)
    return results,raw


print('Local model:',MODEL_PATH,'| Device:',DEVICE)
print('Sequence:',sequence['sequence_id'],'| Text counts:',[len(p) for p in sequence['pages']])


Local model: C:\MODELS\quote-caster-encoder | Device: cuda
Sequence: seq_2fae297a18af2a57 | Text counts: [8, 7, 6]


## 1. Infer entities from ONE page only
Context and clustering are restricted to PAGE_NUMBER. These entity labels are independent of the all-pages run.

In [9]:
one_page_results,one_page_embeddings=infer_entities(page_text_records([PAGE_NUMBER]))

,entry_id,page,text_index,text,entity
0,PG1:T1,1,0,GOT YOUR TOMATO DEVIL RIGHT HERE.,E1
1,PG1:T2,1,1,"IT REVIVES FROM ITS SEEDS, SO YOU SHOULD BURN IT.",E1
2,PG1:T3,1,2,"GOOD WORK, DENJI. THIS GUY'S CORPSE'LL FETCH A...",E1
3,PG1:T4,1,3,THE REWARD'S 400 GRAND.,E1
4,PG1:T5,1,4,THANKS!!,E2
5,PG1:T6,1,5,I'M TAKING OUT 170 GRAND TO COVER YOUR LOAN AN...,E2
6,PG1:T7,1,6,AND THEN THERE'S THE COMMISSION FOR REFERRING ...,E2
7,PG1:T8,1,7,ONLY 70 GRAND LEFT...,E2


Texts: 8 | Anonymous groups: 2 | Raw encoder output: (8, 768)

E1
PG1:T1: GOT YOUR TOMATO DEVIL RIGHT HERE.
PG1:T2: IT REVIVES FROM ITS SEEDS, SO YOU SHOULD BURN IT.
PG1:T3: GOOD WORK, DENJI. THIS GUY'S CORPSE'LL FETCH A PRETTY PENNY ON THE BLACK MARKET.
PG1:T4: THE REWARD'S 400 GRAND.

E2
PG1:T5: THANKS!!
PG1:T6: I'M TAKING OUT 170 GRAND TO COVER YOUR LOAN AND INTEREST PAYMENTS.
PG1:T7: AND THEN THERE'S THE COMMISSION FOR REFERRING YOU AND ASSOCIATED FEES...
PG1:T8: ONLY 70 GRAND LEFT...


In [10]:
print(one_page_embeddings)

[[ 0.01673054 -0.3990283   0.00172448 ...  0.38051566  0.28764522
   0.38400275]
 [-0.08006565 -0.4467661   0.1515682  ...  0.45250663  0.26319632
   0.23008429]
 [-0.01332407 -0.46470904  0.1722949  ...  0.3664256   0.28340054
   0.24497151]
 ...
 [ 0.26277333 -0.16426381  0.6694751  ...  0.24251354  0.26959056
   0.6943888 ]
 [ 0.27357024  0.01495803  0.6628134  ...  0.21189111  0.3058306
   0.56926316]
 [ 0.40346706  0.17444558  0.44019294 ...  0.09163433  0.44005382
   0.6992583 ]]


## 2. Infer entities from ALL pages together
Uses every text in page/entry order, with contextual windows across the combined sequence. Clustering is performed jointly across all pages. No project character mapping is applied.

In [11]:
all_pages_results,all_pages_embeddings=infer_entities(page_text_records(range(1,len(sequence['pages'])+1)))

,entry_id,page,text_index,text,entity
0,PG1:T1,1,0,GOT YOUR TOMATO DEVIL RIGHT HERE.,E1
1,PG1:T2,1,1,"IT REVIVES FROM ITS SEEDS, SO YOU SHOULD BURN IT.",E1
2,PG1:T3,1,2,"GOOD WORK, DENJI. THIS GUY'S CORPSE'LL FETCH A...",E1
3,PG1:T4,1,3,THE REWARD'S 400 GRAND.,E1
4,PG1:T5,1,4,THANKS!!,E2
5,PG1:T6,1,5,I'M TAKING OUT 170 GRAND TO COVER YOUR LOAN AN...,E2
6,PG1:T7,1,6,AND THEN THERE'S THE COMMISSION FOR REFERRING ...,E2
7,PG1:T8,1,7,ONLY 70 GRAND LEFT...,E2
8,PG2:T1,2,0,AFTER PAYING THE WATER BILL,E2
9,PG2:T2,2,1,"AND THE OTHER LOANS,",E2


Texts: 21 | Anonymous groups: 2 | Raw encoder output: (21, 768)

E1
PG1:T1: GOT YOUR TOMATO DEVIL RIGHT HERE.
PG1:T2: IT REVIVES FROM ITS SEEDS, SO YOU SHOULD BURN IT.
PG1:T3: GOOD WORK, DENJI. THIS GUY'S CORPSE'LL FETCH A PRETTY PENNY ON THE BLACK MARKET.
PG1:T4: THE REWARD'S 400 GRAND.
PG2:T6: POCHITA...
PG2:T7: DINNER TODAY'S GONNA BE ONE SLICE OF BREAD.
PG3:T1: WHY'RE YOU USING A BRAT LIKE HIM AS A DEVIL HUNTER?
PG3:T2: TO GET BACK THE MONEY WE'RE OWED.
PG3:T3: THOUGH IT WAS HIS PIECE OF SHIT POPS WHO DIED WITHOUT PAYING UP.
PG3:T4: STILL, IS IT REALLY OKAY TO HIRE A SHITHEAD WITH A PET DEVIL AS A DEVIL HUNTER?
PG3:T5: THING IS, A REAL DEVIL HUNTER, WON'T JUST LET YAKUZA TAKE THE CORPSE AFTERWARDS.
PG3:T6: HIS OTHER REDEEMING QUALITY IS THAT HE ALWAYS DOES AS HE'S TOLD.

E2
PG1:T5: THANKS!!
PG1:T6: I'M TAKING OUT 170 GRAND TO COVER YOUR LOAN AND INTEREST PAYMENTS.
PG1:T7: AND THEN THERE'S THE COMMISSION FOR REFERRING YOU AND ASSOCIATED FEES...
PG1:T8: ONLY 70 GRAND LEFT...
PG2:T1: 